In [0]:
# Databricks notebook source

# ============================================================
# NOTEBOOK DE TESTE END-TO-END DO FRAMEWORK SQL
# VERSAO SEM REDEPLOY DAS PROCS
# - usa as procs já existentes em platform_{env}.utils
# - prepara mocks
# - limpa target corretamente
# - executa os testes
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

# ------------------------------------------------------------
# PARAMETROS
# ------------------------------------------------------------
dbutils.widgets.text("env", "dev")
env_input = dbutils.widgets.get("env")

env_norm = env_input.strip().lower()
if env_norm in ("hmg", "homol", "hml"):
    env_suffix = "hml"
elif env_norm in ("prod", "prd"):
    env_suffix = "prd"
else:
    env_suffix = "dev"

control_catalog = f"platform_{env_suffix}"
lakehouse_catalog = f"lakehouse_{env_suffix}"
test_catalog = "teste_dev"

print(f"env_input       = {env_input}")
print(f"env_suffix      = {env_suffix}")
print(f"control_catalog = {control_catalog}")
print(f"lakehouse_cat   = {lakehouse_catalog}")
print(f"test_catalog    = {test_catalog}")

# ------------------------------------------------------------
# FUNCAO AUXILIAR
# ------------------------------------------------------------
def run_sql(sql_text: str):
    print("--------------------------------------------------")
    print(sql_text)
    return spark.sql(sql_text)

# ------------------------------------------------------------
# 1) CRIAR CATALOGOS E SCHEMAS DE TESTE
# ------------------------------------------------------------
run_sql(f"CREATE CATALOG IF NOT EXISTS {control_catalog}")
run_sql(f"CREATE SCHEMA IF NOT EXISTS {control_catalog}.control")

run_sql(f"CREATE CATALOG IF NOT EXISTS {lakehouse_catalog}")
run_sql(f"CREATE SCHEMA IF NOT EXISTS {lakehouse_catalog}.bronze_mock")
run_sql(f"CREATE SCHEMA IF NOT EXISTS {lakehouse_catalog}.silver_mock")

run_sql(f"CREATE CATALOG IF NOT EXISTS {test_catalog}")
run_sql(f"CREATE SCHEMA IF NOT EXISTS {test_catalog}.scratch")

# ------------------------------------------------------------
# 2) VALIDAR SE AS PROCS EXISTEM
# ------------------------------------------------------------
print("=== CHECK EXISTING PROCEDURES ===")
display(
    spark.sql(f"""
    SHOW PROCEDURES IN {control_catalog}.utils
    """)
)

# ------------------------------------------------------------
# 3) LIMPAR OBJETOS DE TESTE
# ------------------------------------------------------------
for obj in [
    f"{control_catalog}.control.cfg_table_constraints",
    f"{control_catalog}.control.cfg_ingestion_state",
    f"{control_catalog}.control.cfg_target",
    f"{control_catalog}.control.cfg_source",
    f"{control_catalog}.control.cfg_config",
    f"{lakehouse_catalog}.bronze_mock.customer_src",
    f"{lakehouse_catalog}.silver_mock.customer_dim",
]:
    try:
        run_sql(f"DROP TABLE IF EXISTS {obj}")
    except Exception as e:
        print(f"warning dropping {obj}: {e}")

target_path = f"abfss://silver@stdatacloud{env_suffix}dblhbrs001.dfs.core.windows.net/mockdomain/mocksys/customer_dim"

try:
    dbutils.fs.rm(target_path, True)
    print(f"initial cleanup path removed: {target_path}")
except Exception as e:
    print(f"initial cleanup path skipped: {e}")

# ------------------------------------------------------------
# 4) CRIAR TABELAS DE CONTROLE MOCK
# ------------------------------------------------------------
run_sql(f"""
CREATE TABLE IF NOT EXISTS {control_catalog}.control.cfg_config
(
  config_id    STRING,
  project      STRING,
  domain       STRING,
  system       STRING,
  layer        STRING,
  config_name  STRING,
  is_active    BOOLEAN,
  created_at   TIMESTAMP,
  updated_at   TIMESTAMP
)
USING DELTA
""")

run_sql(f"""
CREATE TABLE IF NOT EXISTS {control_catalog}.control.cfg_source
(
  config_id                 STRING,
  source_type               STRING,
  source_format             STRING,
  source_catalog            STRING,
  source_schema             STRING,
  source_object             STRING,
  source_query              STRING,
  source_path               STRING,
  load_mode                 STRING,
  watermark_column          STRING,
  watermark_initial_value   STRING,
  lookback_days             INT,
  created_at                TIMESTAMP,
  updated_at                TIMESTAMP
)
USING DELTA
""")

run_sql(f"""
CREATE TABLE IF NOT EXISTS {control_catalog}.control.cfg_target
(
  config_id                 STRING,
  target_type               STRING,
  target_format             STRING,
  target_catalog            STRING,
  target_schema             STRING,
  target_object             STRING,
  write_mode                STRING,
  business_keys             ARRAY<STRING>,
  clustering_columns        ARRAY<STRING>,
  target_schema_definition  MAP<STRING, STRING>,
  created_at                TIMESTAMP,
  updated_at                TIMESTAMP
)
USING DELTA
""")

run_sql(f"""
CREATE TABLE IF NOT EXISTS {control_catalog}.control.cfg_ingestion_state
(
  config_id       STRING,
  state_type      STRING,
  last_value      STRING,
  last_run_ts     TIMESTAMP,
  status          STRING,
  error_message   STRING
)
USING DELTA
""")

run_sql(f"""
CREATE TABLE IF NOT EXISTS {control_catalog}.control.cfg_table_constraints
(
  config_id          STRING,
  constraint_name    STRING,
  constraint_type    STRING,
  column_name        STRING,
  reference_schema   STRING,
  reference_table    STRING,
  reference_column   STRING,
  expression         STRING,
  is_active          BOOLEAN
)
USING DELTA
""")

# ------------------------------------------------------------
# 5) CRIAR TABELA BRONZE MOCK
# ------------------------------------------------------------
run_sql(f"""
CREATE TABLE {lakehouse_catalog}.bronze_mock.customer_src
(
  customer_id         INT,
  customer_name       STRING,
  city                STRING,
  updated_at          TIMESTAMP,
  _hashkey            STRING,
  _ingestion_bronze   TIMESTAMP
)
USING DELTA
""")

run_sql(f"""
INSERT INTO {lakehouse_catalog}.bronze_mock.customer_src VALUES
(1, 'Alice', 'Curitiba', timestamp('2026-03-20 10:00:00'), 'hk_1_v1', timestamp('2026-03-24 10:00:00')),
(1, 'Alice', 'Curitiba', timestamp('2026-03-21 10:00:00'), 'hk_1_v2', timestamp('2026-03-24 11:00:00')),
(2, 'Bruno', 'Toledo',   timestamp('2026-03-20 09:30:00'), 'hk_2_v1', timestamp('2026-03-24 10:30:00')),
(3, 'Carla', 'Cascavel', timestamp('2026-03-19 08:00:00'), 'hk_3_v1', timestamp('2026-03-24 09:00:00'))
""")

# ------------------------------------------------------------
# 6) INSERIR CONFIG MOCK
# contrato final já com _hashkey
# ------------------------------------------------------------
run_sql(f"""
INSERT INTO {control_catalog}.control.cfg_config VALUES
(
  'cfg_silver_mock_customer_dim',
  'mock_project',
  'mockdomain',
  'mocksys',
  'SILVER',
  'customer_dim',
  true,
  current_timestamp(),
  current_timestamp()
)
""")

run_sql(f"""
INSERT INTO {control_catalog}.control.cfg_source VALUES
(
  'cfg_silver_mock_customer_dim',
  'TABLE',
  'DELTA',
  'lakehouse',
  'bronze_mock',
  'customer_src',
  NULL,
  NULL,
  'FULL',
  NULL,
  NULL,
  0,
  current_timestamp(),
  current_timestamp()
)
""")

run_sql(f"""
INSERT INTO {control_catalog}.control.cfg_target VALUES
(
  'cfg_silver_mock_customer_dim',
  'TABLE',
  'DELTA',
  'lakehouse',
  'silver_mock',
  'customer_dim',
  'MERGE',
  array('customer_id'),
  array(),
  map(
    'customer_id', 'INT',
    'customer_name', 'STRING',
    'city', 'STRING',
    'updated_at', 'TIMESTAMP',
    '_hashkey', 'STRING'
  ),
  current_timestamp(),
  current_timestamp()
)
""")

run_sql(f"""
INSERT INTO {control_catalog}.control.cfg_ingestion_state VALUES
(
  'cfg_silver_mock_customer_dim',
  'WATERMARK',
  NULL,
  current_timestamp(),
  'SUCCESS',
  NULL
)
""")

# ------------------------------------------------------------
# 7) VALIDACAO RAPIDA DO CFG
# ------------------------------------------------------------
print("=== TEST 1: proc_load_config ===")
display(
    spark.sql(f"""
    CALL {control_catalog}.utils.proc_load_config(
      p_config_name => 'customer_dim',
      p_layer       => 'SILVER',
      p_system      => 'mocksys'
    )
    """)
)

print("=== TEST 2: cfg temp view ===")
display(spark.sql("SELECT * FROM cfg"))

# ------------------------------------------------------------
# 8) LIMPEZA FINAL DO TARGET ANTES DO TESTE PRINCIPAL
# ------------------------------------------------------------
print("=== CLEAN TARGET BEFORE TEST 5 ===")

spark.sql(f"DROP TABLE IF EXISTS {lakehouse_catalog}.silver_mock.customer_dim")

try:
    dbutils.fs.rm(target_path, True)
    print("Path removido com sucesso")
except Exception as e:
    print("Erro ao remover path:", e)

try:
    dbutils.fs.ls(target_path)
    print("WARNING: PATH AINDA EXISTE")
except Exception as e:
    print("OK: PATH LIMPO", e)

print("=== CHECK TABLES REGISTERED BEFORE TEST 5 ===")
display(
    spark.sql(f"""
    SELECT table_catalog, table_schema, table_name
    FROM system.information_schema.tables
    WHERE LOWER(table_name) = 'customer_dim'
    ORDER BY table_catalog, table_schema
    """)
)

# ------------------------------------------------------------
# 9) TESTE PRINCIPAL
# usa proc existente
# ------------------------------------------------------------
print("=== TEST 5: proc_run_silver_scd1 ===")
display(
    spark.sql(f"""
    CALL {control_catalog}.utils.proc_run_silver_scd1(
      p_config_name => 'customer_dim',
      p_layer       => 'SILVER',
      p_system      => 'mocksys',
      p_env         => '{env_suffix}'
    )
    """)
)

print("=== TEST 6: resultado da target após primeira carga ===")
display(
    spark.sql(f"""
    SELECT *
    FROM {lakehouse_catalog}.silver_mock.customer_dim
    ORDER BY customer_id
    """)
)

# ------------------------------------------------------------
# 10) TESTE DE UPDATE SCD1
# ------------------------------------------------------------
run_sql(f"""
INSERT INTO {lakehouse_catalog}.bronze_mock.customer_src VALUES
(2, 'Bruno Silva', 'Maringá', timestamp('2026-03-24 12:00:00'), 'hk_2_v2', timestamp('2026-03-24 12:30:00'))
""")

print("=== TEST 7: executar merge novamente ===")
display(
    spark.sql(f"""
    CALL {control_catalog}.utils.proc_run_silver_scd1(
      p_config_name => 'customer_dim',
      p_layer       => 'SILVER',
      p_system      => 'mocksys',
      p_env         => '{env_suffix}'
    )
    """)
)

print("=== TEST 8: resultado final esperado ===")
final_df = spark.sql(f"""
SELECT *
FROM {lakehouse_catalog}.silver_mock.customer_dim
ORDER BY customer_id
""")
display(final_df)

# ------------------------------------------------------------
# 11) ASSERTIVAS
# ------------------------------------------------------------
rows = final_df.collect()

assert len(rows) == 3, f"Esperado 3 registros na silver, obtido: {len(rows)}"

row1 = [r for r in rows if r["customer_id"] == 1][0]
row2 = [r for r in rows if r["customer_id"] == 2][0]
row3 = [r for r in rows if r["customer_id"] == 3][0]

assert row1["customer_name"] == "Alice"
assert row1["city"] == "Curitiba"
assert row1["_hashkey"] == "hk_1_v2"

assert row2["customer_name"] == "Bruno Silva"
assert row2["city"] == "Maringá"
assert row2["_hashkey"] == "hk_2_v2"

assert row3["customer_name"] == "Carla"
assert row3["city"] == "Cascavel"
assert row3["_hashkey"] == "hk_3_v1"

print("ALL ASSERTS PASSED")

# ------------------------------------------------------------
# 12) PROVAS DE SUFIXO POR ENV
# ------------------------------------------------------------
print("=== TEST 9: conferir catálogo real da target ===")
display(
    spark.sql(f"""
    SELECT
      table_catalog,
      table_schema,
      table_name
    FROM system.information_schema.tables
    WHERE LOWER(table_schema) = 'silver_mock'
      AND LOWER(table_name) = 'customer_dim'
    ORDER BY table_catalog
    """)
)

print("=== TEST 10: conferir conteúdo final ===")
display(
    spark.sql(f"""
    SELECT *
    FROM {lakehouse_catalog}.silver_mock.customer_dim
    ORDER BY customer_id
    """)
)

print("Teste concluído com sucesso.")